# Modelado — Predicción de Malnutrición a 12 meses EC
## PMCI / Fundación Canguro

**Estrategia**: Cascada temporal con LightGBM
- **F0** → solo prenatal/parto
- **F1** → + nacimiento
- **F2** → + hospitalización
- **F3** → + 40 semanas
- **F4** → + 3 meses
- **F5** → + 6 meses
- **F6** → + 9 meses

**Outcomes**: Stunting (HAZ<-2) | Bajo peso (WAZ<-2) | Wasting (WHZ<-2)

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import shap
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

PATH = 'KMC-70k-93-2024-Malnutricion-conVel-DATA-SPSS-20250322.xlsx'
OUT  = '/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/'
PLAN = 'feature_plan.json'

print('Cargando datos...')
df_raw = pd.read_excel(PATH)
df = df_raw.replace('#NULL!', np.nan).copy()
for col in df.columns:
    c = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() == 0 or c.notna().sum() / df[col].notna().sum() >= 0.5:
        df[col] = c

with open(PLAN) as f:
    plan = json.load(f)

FASES = plan['fases']
print(f'Dataset: {df.shape[0]:,} x {df.shape[1]:,}')
print(f'Fases cargadas: {list(FASES.keys())}')

## 1. Preprocesamiento y Variables Objetivo

In [2]:
# Variables objetivo binarias
df['stunting12m']      = np.where(df['zscoretalla12cat'].notna(),
                                   (df['zscoretalla12cat'] == 1.0).astype(float), np.nan)
df['underweight12m_b'] = np.where(df['zscorepeso12cat'].notna(),
                                   (df['zscorepeso12cat'] == 1.0).astype(float), np.nan)
df['wasting12m']       = np.where(df['zscorepesotalla12cat'].notna(),
                                   (df['zscorepesotalla12cat'] == 1.0).astype(float), np.nan)

OUTCOMES = {
    'Stunting':    ('stunting12m',      '#e74c3c', 3.1),
    'Bajo_peso':   ('underweight12m_b', '#e67e22', 8.2),
    'Wasting':     ('wasting12m',        '#f39c12', 23.2),
}

# Variables leakage (derivadas de datos de 12m o posteriores)
LEAKAGE = {
    'velocidad12_9mesesOMS',   # velocidad 9→12m (requiere dato de 12m)
    'vino12m', 'Desercionreal12meses',
    'rehosp40a12meses', 'mortalidad40sem12meses',
    'indexnutricion12meses', 'MUERTE1ANO',
    'examenneurodurante12meses', 'examenneuropsico12meses',
    'riesgoPC12m',
}
# Todas las variables con '12' en el nombre son del futuro
LEAKAGE |= {c for c in df.columns if '12' in str(c) and c not in
            ['stunting12m','underweight12m_b','wasting12m']}

# Construir feature sets acumulados por fase (cascada)
FASE_ORDER = ['F0_Prenatal_Parto','F1_Nacimiento','F2_Hospitalizacion',
              'F3_40semanas','F4_3meses','F5_6meses','F6_9meses']

cumulative_features = {}
acum = []
for fase in FASE_ORDER:
    cols = FASES.get(fase, [])
    nuevas = [c for c in cols if c in df.columns and c not in LEAKAGE]
    acum = acum + [c for c in nuevas if c not in acum]
    cumulative_features[fase] = list(acum)

print('Feature sets acumulados por fase:')
for fase, cols in cumulative_features.items():
    print(f'  {fase}: {len(cols)} features')

Feature sets acumulados por fase:
  F0_Prenatal_Parto: 41 features
  F1_Nacimiento: 75 features
  F2_Hospitalizacion: 107 features
  F3_40semanas: 137 features
  F4_3meses: 164 features
  F5_6meses: 183 features
  F6_9meses: 198 features


## 1b. Clasificación Compuesta de Estado Nutricional a 12 meses EC

Construcción de un outcome multiclase que combina los tres indicadores OMS en grupos clínicamente interpretables.
Esta clasificación será validada con los neonatólogos de la Fundación Canguro.

In [ ]:
# --- Construcción del outcome compuesto (4 grupos) ---
# Requiere que stunting12m, underweight12m_b, wasting12m ya estén definidas

# Número de déficits simultáneos por paciente
df['n_deficits'] = (
    df['stunting12m'].fillna(0) +
    df['underweight12m_b'].fillna(0) +
    df['wasting12m'].fillna(0)
)

# Sobrepeso/obesidad (WHZ o WAZ > +2) — variable existente en el dataset
df['overweight12m'] = pd.to_numeric(df.get('Overweightorobesity12m', np.nan), errors='coerce')

# Clasificación compuesta — 4 grupos mutuamente excluyentes
# Prioridad: desnutrición múltiple > un déficit > sobrepeso > normal
def clasificar_estado_nutricional(row):
    # Solo clasificar si hay al menos un outcome no nulo
    if pd.isna(row['stunting12m']) and pd.isna(row['underweight12m_b']) and pd.isna(row['wasting12m']):
        return np.nan
    n = row['n_deficits']
    if n >= 2:
        return 3   # Desnutrición múltiple (2-3 déficits)
    elif n == 1:
        return 2   # Un déficit
    elif row['overweight12m'] == 1:
        return 1   # Sobrepeso/obesidad sin desnutrición
    else:
        return 0   # Normal / Crecimiento armónico

df['estado_nutricional_12m'] = df.apply(clasificar_estado_nutricional, axis=1)

GRUPO_LABELS = {
    0: 'Normal',
    1: 'Sobrepeso/Obesidad',
    2: 'Un déficit',
    3: 'Desnutrición múltiple'
}

# Distribución general
print('=' * 60)
print('CLASIFICACIÓN COMPUESTA — Estado Nutricional a 12 meses EC')
print('=' * 60)

conteo = df['estado_nutricional_12m'].value_counts().sort_index()
total  = conteo.sum()
for grupo, n in conteo.items():
    print(f'  Grupo {int(grupo)} — {GRUPO_LABELS[int(grupo)]:<25}: {n:>6,}  ({n/total:.1%})')
print(f'  {"Sin dato":<32}: {df["estado_nutricional_12m"].isna().sum():>6,}')
print(f'  {"TOTAL con clasificación":<32}: {total:>6,}')

# Comparar con indexnutricion12meses existente
print('\nComparación con indexnutricion12meses (variable original):')
comp = pd.crosstab(
    df['estado_nutricional_12m'].map(GRUPO_LABELS).fillna('Sin dato'),
    df['indexnutricion12meses'].map({1: 'Armónico', 0: 'No armónico'}).fillna('Sin dato'),
    margins=True
)
print(comp.to_string())

In [ ]:
# --- Visualización de la distribución por grupos ---

GRUPO_COLORS = {
    'Normal':                '#27ae60',
    'Sobrepeso/Obesidad':    '#f39c12',
    'Un déficit':            '#e67e22',
    'Desnutrición múltiple': '#e74c3c',
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: Distribución general
ax1 = axes[0]
labels = [GRUPO_LABELS[i] for i in sorted(GRUPO_LABELS)]
values = [conteo.get(i, 0) for i in sorted(GRUPO_LABELS)]
colors = [GRUPO_COLORS[l] for l in labels]
bars   = ax1.bar(labels, values, color=colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{v:,}\n({v/total:.1%})', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_ylabel('N pacientes')
ax1.set_title('Distribución de grupos\nnutricionales a 12 meses EC', fontweight='bold')
ax1.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')

# Panel 2: Composición interna — qué déficits tiene cada grupo
ax2 = axes[1]
deficit_cols = ['stunting12m', 'underweight12m_b', 'wasting12m']
deficit_names = ['Stunting\n(HAZ<-2)', 'Bajo peso\n(WAZ<-2)', 'Wasting\n(WHZ<-2)']

x  = np.arange(len(deficit_names))
w  = 0.22
grupos_plot = [2, 3]  # "Un déficit" y "Desnutrición múltiple"
offsets = [-0.11, 0.11]

for offset, grupo_id in zip(offsets, grupos_plot):
    sub   = df[df['estado_nutricional_12m'] == grupo_id]
    prevs = [sub[c].mean() for c in deficit_cols]
    label = GRUPO_LABELS[grupo_id]
    ax2.bar(x + offset, prevs, width=w,
            color=GRUPO_COLORS[label], alpha=0.85, label=label)

ax2.set_xticks(x); ax2.set_xticklabels(deficit_names, fontsize=9)
ax2.set_ylabel('Prevalencia')
ax2.set_ylim(0, 1.05)
ax2.set_title('Prevalencia de cada déficit\npor grupo nutricional', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3, axis='y')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Panel 3: Distribución por cohorte temporal (P4, P5, P6)
ax3 = axes[2]
if 'cohort' in df.columns:
    df_coh = df[df['cohort'].notna() & df['estado_nutricional_12m'].notna()]
    cohort_group = df_coh.groupby(['cohort', 'estado_nutricional_12m']).size().unstack(fill_value=0)
    cohort_pct   = cohort_group.div(cohort_group.sum(axis=1), axis=0)
    bottom = np.zeros(len(cohort_pct))
    for grupo_id in sorted(GRUPO_LABELS):
        if grupo_id in cohort_pct.columns:
            vals = cohort_pct[grupo_id].values
            ax3.bar(cohort_pct.index, vals, bottom=bottom,
                    color=GRUPO_COLORS[GRUPO_LABELS[grupo_id]],
                    label=GRUPO_LABELS[grupo_id], edgecolor='white')
            bottom += vals
    ax3.set_ylabel('Proporción')
    ax3.set_title('Composición nutricional\npor cohorte temporal', fontweight='bold')
    ax3.legend(fontsize=7, loc='upper right')
    ax3.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax3.set_xticklabels(cohort_pct.index, rotation=15, ha='right', fontsize=8)
    ax3.grid(True, alpha=0.3, axis='y')
else:
    ax3.text(0.5, 0.5, 'Ejecutar sección 9\npara cohortes temporales',
             ha='center', va='center', transform=ax3.transAxes, fontsize=10)
    ax3.axis('off')

plt.suptitle('Estado Nutricional Compuesto a 12 meses EC — PMCI Fundación Canguro',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla para revisión con neonatólogos
print('\nTABLA PARA VALIDACIÓN CON EXPERTOS:')
print('=' * 65)
print(f'{"Grupo":<30} {"N":>6}  {"% del total":>10}  {"Criterio"}')
print('-' * 65)
criterios = {
    0: 'HAZ≥-2  AND  WAZ≥-2  AND  WHZ≥-2',
    1: 'Sobrepeso sin desnutrición',
    2: 'Exactamente 1 de: HAZ<-2, WAZ<-2, WHZ<-2',
    3: '2 o más de: HAZ<-2, WAZ<-2, WHZ<-2',
}
for i in sorted(GRUPO_LABELS):
    n = conteo.get(i, 0)
    print(f'  {GRUPO_LABELS[i]:<28} {n:>6,}  {n/total:>10.1%}  {criterios[i]}')
print('=' * 65)
print('\n⚠  PENDIENTE: validar esta clasificación con neonatólogos (Charpak / Lince)')

## 2. Funciones de Entrenamiento y Evaluación

In [3]:
def get_model_data(df, features, target_col, min_samples=100):
    """Retorna X, y con solo filas que tienen el outcome."""
    cols = [c for c in features if c in df.columns]
    sub  = df[cols + [target_col]].dropna(subset=[target_col]).copy()
    X    = sub[cols]
    y    = sub[target_col].astype(int)
    return X, y


def train_lgbm_cv(X, y, n_splits=5, spw=None, seed=42):
    """
    Entrena LightGBM con validacion cruzada estratificada.
    Retorna metricas por fold y predicciones OOF (Out-Of-Fold).
    """
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    scale_pos_weight = spw if spw else round(n_neg / n_pos, 2)

    params = {
        'objective':         'binary',
        'metric':            'auc',
        'learning_rate':     0.05,
        'num_leaves':        63,
        'max_depth':         -1,
        'min_child_samples': 30,
        'feature_fraction':  0.8,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'scale_pos_weight':  scale_pos_weight,
        'verbose':           -1,
        'seed':              seed,
    }

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics = []
    oof_pred = np.zeros(len(y))
    models  = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            params, dtrain,
            num_boost_round=500,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(-1),
            ]
        )
        models.append(model)

        prob = model.predict(X_val)
        oof_pred[val_idx] = prob

        auc = roc_auc_score(y_val, prob)
        thr = 0.5
        pred_bin = (prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred_bin).ravel()
        metrics.append({
            'fold':        fold + 1,
            'AUC':         round(auc, 4),
            'Sens':        round(tp / (tp+fn) if tp+fn > 0 else 0, 4),
            'Spec':        round(tn / (tn+fp) if tn+fp > 0 else 0, 4),
            'F1':          round(f1_score(y_val, pred_bin, zero_division=0), 4),
            'Precision':   round(precision_score(y_val, pred_bin, zero_division=0), 4),
            'n_train':     len(y_tr),
            'n_val':       len(y_val),
            'best_iter':   model.best_iteration,
        })

    return pd.DataFrame(metrics), oof_pred, models


print('Funciones definidas: get_model_data, train_lgbm_cv')

Funciones definidas: get_model_data, train_lgbm_cv


## 3. Cascada Temporal — Todos los Outcomes

In [4]:
# Entrenar LightGBM para cada fase × cada outcome
# Almacena resultados, predicciones OOF y mejores modelos

resultados  = {}   # (outcome, fase) -> DataFrame de metricas por fold
oof_preds   = {}   # (outcome, fase) -> array de probabilidades OOF
best_models = {}   # (outcome, fase) -> lista de modelos (folds)
data_store  = {}   # (outcome, fase) -> (X, y)

for outcome_name, (target_col, color, _) in OUTCOMES.items():
    print(f'\n========== {outcome_name} ==========')
    for fase in FASE_ORDER:
        feats = cumulative_features[fase]
        X, y  = get_model_data(df, feats, target_col)

        if len(y) < 200 or y.sum() < 20:
            print(f'  {fase}: insuficientes datos (n={len(y)}, pos={y.sum()}) — omitida')
            continue

        metrics_df, oof, models = train_lgbm_cv(X, y)
        auc_mean = metrics_df['AUC'].mean()
        auc_std  = metrics_df['AUC'].std()

        resultados [(outcome_name, fase)] = metrics_df
        oof_preds  [(outcome_name, fase)] = (oof, y)
        best_models[(outcome_name, fase)] = models
        data_store [(outcome_name, fase)] = (X, y)

        sens_m = metrics_df['Sens'].mean()
        spec_m = metrics_df['Spec'].mean()
        print(f'  {fase:<22}: AUC={auc_mean:.4f}±{auc_std:.4f}  '
              f'Sens={sens_m:.3f}  Spec={spec_m:.3f}  n={len(y):,}')


========== Stunting ==========
  F0_Prenatal_Parto     : AUC=0.6454±0.0102  Sens=0.509  Spec=0.689  n=30,953
  F1_Nacimiento         : AUC=0.7374±0.0056  Sens=0.641  Spec=0.708  n=30,953
  F2_Hospitalizacion    : AUC=0.7405±0.0063  Sens=0.639  Spec=0.713  n=30,953
  F3_40semanas          : AUC=0.7678±0.0081  Sens=0.621  Spec=0.758  n=30,953
  F4_3meses             : AUC=0.8209±0.0094  Sens=0.691  Spec=0.779  n=30,953
  F5_6meses             : AUC=0.8935±0.0039  Sens=0.792  Spec=0.823  n=30,953
  F6_9meses             : AUC=0.9290±0.0031  Sens=0.834  Spec=0.859  n=30,953

========== Bajo_peso ==========
  F0_Prenatal_Parto     : AUC=0.6183±0.0081  Sens=0.396  Spec=0.749  n=29,897
  F1_Nacimiento         : AUC=0.7509±0.0116  Sens=0.564  Spec=0.780  n=29,897
  F2_Hospitalizacion    : AUC=0.7538±0.0115  Sens=0.550  Spec=0.796  n=29,897
  F3_40semanas          : AUC=0.7725±0.0118  Sens=0.547  Spec=0.827  n=29,897
  F4_3meses             : AUC=0.8737±0.0083  Sens=0.685  Spec=0.867  n=29,897

## 4. Comparación de AUC por Fase y Outcome

In [5]:
# Tabla resumen de AUC media por fase y outcome
rows = []
for (outcome, fase), mdf in resultados.items():
    rows.append({
        'Outcome': outcome,
        'Fase':    fase,
        'AUC_mean':  mdf['AUC'].mean(),
        'AUC_std':   mdf['AUC'].std(),
        'Sens_mean': mdf['Sens'].mean(),
        'Spec_mean': mdf['Spec'].mean(),
        'F1_mean':   mdf['F1'].mean(),
        'N_total':   resultados[(outcome,fase)]['n_train'].iloc[0] +
                     resultados[(outcome,fase)]['n_val'].iloc[0],
    })

summary = pd.DataFrame(rows)
pivot_auc = summary.pivot(index='Fase', columns='Outcome', values='AUC_mean')
# Reordenar filas por fase
fase_order_present = [f for f in FASE_ORDER if f in pivot_auc.index]
pivot_auc = pivot_auc.loc[fase_order_present]

print('AUC media por fase y outcome (5-fold CV):')
print(pivot_auc.round(4).to_string())

# Tabla completa de metricas
print('\nTabla completa de metricas:')
tbl = summary[['Outcome','Fase','AUC_mean','AUC_std','Sens_mean','Spec_mean','F1_mean']]
tbl = tbl.round(4)
print(tbl.to_string(index=False))

AUC media por fase y outcome (5-fold CV):
Outcome             Bajo_peso  Stunting  Wasting
Fase                                            
F0_Prenatal_Parto      0.6183    0.6454   0.5550
F1_Nacimiento          0.7509    0.7374   0.6887
F2_Hospitalizacion     0.7538    0.7405   0.7054
F3_40semanas           0.7725    0.7678   0.7253
F4_3meses              0.8737    0.8209   0.8260
F5_6meses              0.9360    0.8935   0.8950
F6_9meses              0.9634    0.9290   0.9245

Tabla completa de metricas:
  Outcome               Fase  AUC_mean  AUC_std  Sens_mean  Spec_mean  F1_mean
 Stunting  F0_Prenatal_Parto    0.6454   0.0102     0.5088     0.6891   0.4133
 Stunting      F1_Nacimiento    0.7374   0.0056     0.6406     0.7077   0.5054
 Stunting F2_Hospitalizacion    0.7405   0.0063     0.6391     0.7135   0.5080
 Stunting       F3_40semanas    0.7678   0.0081     0.6214     0.7583   0.5262
 Stunting          F4_3meses    0.8209   0.0094     0.6911     0.7793   0.5838
 Stunting     

In [ ]:
# Grafica: evolucion del AUC a lo largo de la cascada
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fase_labels_short = {
    'F0_Prenatal_Parto':  'F0\nPrenatal',
    'F1_Nacimiento':      'F1\nNacimiento',
    'F2_Hospitalizacion': 'F2\nHosp.',
    'F3_40semanas':       'F3\n40 sem',
    'F4_3meses':          'F4\n3m',
    'F5_6meses':          'F5\n6m',
    'F6_9meses':          'F6\n9m',
}

for ax, (outcome_name, (_, color, _)) in zip(axes, OUTCOMES.items()):
    fases_ok = [f for f in FASE_ORDER if (outcome_name, f) in resultados]
    aucs     = [resultados[(outcome_name, f)]['AUC'].mean() for f in fases_ok]
    stds     = [resultados[(outcome_name, f)]['AUC'].std()  for f in fases_ok]
    x_lbls   = [fase_labels_short[f] for f in fases_ok]
    x        = np.arange(len(fases_ok))

    ax.plot(x, aucs, marker='o', linewidth=2.5, markersize=8, color=color)
    ax.fill_between(x,
                    np.array(aucs) - np.array(stds),
                    np.array(aucs) + np.array(stds),
                    alpha=0.15, color=color)
    for xi, auc_v, std_v in zip(x, aucs, stds):
        ax.text(xi, auc_v + 0.008, f'{auc_v:.3f}',
                ha='center', fontsize=9, fontweight='bold', color=color)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Aleatorio')
    ax.set_xticks(x)
    ax.set_xticklabels(x_lbls, fontsize=9)
    ax.set_ylim(0.45, 1.0)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'{outcome_name}\n¿Cuando puede predecirse?', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Evolucion del ROC-AUC por fase temporal — Cascada de prediccion',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Grafica: AUC, Sensibilidad, Especificidad para el mejor outcome (Stunting)
outcome_focus = 'Stunting'
color_focus   = '#e74c3c'

fases_ok = [f for f in FASE_ORDER if (outcome_focus, f) in resultados]
x_lbls   = [fase_labels_short[f] for f in fases_ok]
x        = np.arange(len(fases_ok))

met_rows = {
    'AUC':          [resultados[(outcome_focus,f)]['AUC'].mean()  for f in fases_ok],
    'Sensibilidad': [resultados[(outcome_focus,f)]['Sens'].mean() for f in fases_ok],
    'Especificidad':[resultados[(outcome_focus,f)]['Spec'].mean() for f in fases_ok],
    'F1':           [resultados[(outcome_focus,f)]['F1'].mean()   for f in fases_ok],
}

fig, ax = plt.subplots(figsize=(13, 5))
met_colors = ['#e74c3c','#3498db','#2ecc71','#f39c12']
for (met, vals), col in zip(met_rows.items(), met_colors):
    ax.plot(x, vals, marker='o', linewidth=2, markersize=7, color=col, label=met)

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(x_lbls, fontsize=10)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('Metrica')
ax.set_title(f'Metricas por fase — {outcome_focus}\n'
             f'(AUC, Sensibilidad, Especificidad, F1)', fontweight='bold')
ax.legend(ncol=4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC por fase para Stunting (usando predicciones OOF)
fig, ax = plt.subplots(figsize=(8, 7))

palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(fases_ok)))

for fase, col in zip(fases_ok, palette):
    oof, y_true = oof_preds[(outcome_focus, fase)]
    fpr, tpr, _ = roc_curve(y_true, oof)
    auc_v = roc_auc_score(y_true, oof)
    ax.plot(fpr, tpr, color=col, linewidth=2,
            label=f'{fase_labels_short[fase].replace(chr(10)," ")} (AUC={auc_v:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Aleatorio')
ax.set_xlabel('1 - Especificidad (FPR)')
ax.set_ylabel('Sensibilidad (TPR)')
ax.set_title(f'Curvas ROC por fase — {outcome_focus}\n(predicciones OOF)', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas Precision-Recall (mas informativa con desbalance de clases)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (outcome_name, (_, color, _)) in zip(axes, OUTCOMES.items()):
    fases_out = [f for f in FASE_ORDER if (outcome_name, f) in oof_preds]
    palette_o = plt.cm.plasma(np.linspace(0.1, 0.9, len(fases_out)))

    for fase, col in zip(fases_out, palette_o):
        oof, y_true = oof_preds[(outcome_name, fase)]
        prec, rec, _ = precision_recall_curve(y_true, oof)
        ap = average_precision_score(y_true, oof)
        ax.plot(rec, prec, color=col, linewidth=1.8,
                label=f'{fase_labels_short[fase].replace(chr(10)," ")} (AP={ap:.3f})')

    baseline = y_true.mean()
    ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5,
               label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel('Recall (Sensibilidad)')
    ax.set_ylabel('Precision')
    ax.set_title(f'{outcome_name}\nPrecision-Recall por fase', fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Curvas Precision-Recall por fase y outcome (OOF)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Interpretabilidad — SHAP Values

In [11]:
# Identificar la mejor fase por outcome para análisis SHAP
best_fase_per_outcome = {}
for outcome_name in OUTCOMES:
    best_auc  = -1
    best_fase = None
    for fase in FASE_ORDER:
        if (outcome_name, fase) in resultados:
            auc = resultados[(outcome_name, fase)]['AUC'].mean()
            if auc > best_auc:
                best_auc  = auc
                best_fase = fase
    best_fase_per_outcome[outcome_name] = (best_fase, best_auc)
    print(f'{outcome_name}: mejor fase = {best_fase} (AUC={best_auc:.4f})')

# Tambien calcular SHAP para F1 (solo nacimiento) — valor clinico maximo
print('\nFases elegidas para SHAP:')
print('  - Mejor fase por AUC (por outcome)')
print('  - F1_Nacimiento (para comparar con solo datos de nacimiento)')

Stunting: mejor fase = F6_9meses (AUC=0.9290)
Bajo_peso: mejor fase = F6_9meses (AUC=0.9634)
Wasting: mejor fase = F6_9meses (AUC=0.9245)

Fases elegidas para SHAP:
  - Mejor fase por AUC (por outcome)
  - F1_Nacimiento (para comparar con solo datos de nacimiento)


In [12]:
# SHAP para Stunting — mejor fase
outcome_shap = 'Stunting'
best_fase_shap, _ = best_fase_per_outcome[outcome_shap]
X_shap, y_shap   = data_store[(outcome_shap, best_fase_shap)]

# Usar el primer modelo del fold (entrenado en ~80% de datos)
model_shap = best_models[(outcome_shap, best_fase_shap)][0]

print(f'Calculando SHAP para {outcome_shap} — {best_fase_shap}')
print(f'  Dataset: {X_shap.shape[0]:,} muestras x {X_shap.shape[1]} features')

# Muestra representativa para SHAP (max 3000 filas por velocidad)
n_shap = min(3000, len(X_shap))
X_sample = X_shap.sample(n_shap, random_state=42)

explainer   = shap.TreeExplainer(model_shap)
shap_values = explainer.shap_values(X_sample)

print(f'  SHAP values calculados: shape={np.array(shap_values).shape}')

Calculando SHAP para Stunting — F6_9meses
  Dataset: 30,953 muestras x 198 features
  SHAP values calculados: shape=(3000, 198)


In [ ]:
# SHAP Summary Plot — Top 20 factores de riesgo
shap.summary_plot(
    shap_values, X_sample,
    max_display=20,
    plot_type='dot',
    show=False
)
plt.title(f'SHAP — Top 20 factores de riesgo\n'
          f'{outcome_shap} | {best_fase_shap}\n'
          f'(rojo=aumenta riesgo, azul=disminuye riesgo)',
          fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar plot — importancia global de features
shap_importance = pd.DataFrame({
    'feature':       X_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors_shap = ['#e74c3c' if i < 5 else '#e67e22' if i < 10 else '#3498db'
               for i in range(len(shap_importance))]
bars = ax.barh(shap_importance['feature'][::-1],
               shap_importance['mean_abs_shap'][::-1],
               color=colors_shap[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP value| (impacto promedio en la prediccion)')
ax.set_title(f'Importancia global de features — SHAP\n{outcome_shap} | {best_fase_shap}',
             fontweight='bold')

for bar, v in zip(bars, shap_importance['mean_abs_shap'][::-1]):
    ax.text(v + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print('Top 10 factores de riesgo (SHAP):')
for _, row in shap_importance.head(10).iterrows():
    print(f'  {row["feature"]:<40} SHAP={row["mean_abs_shap"]:.4f}')

## 6. Sistema de Riesgo Dinámico
Visualización de cómo evoluciona la probabilidad de riesgo de un paciente a través de las fases.

In [17]:
# Calcular probabilidades en cada fase para todos los pacientes
# Solo para Stunting (outcome principal)
# Para cada paciente que tenga datos en TODAS las fases, calculamos la trayectoria

outcome_dyn = 'Stunting'
target_dyn  = OUTCOMES[outcome_dyn][0]

# Pacientes con outcome conocido
df_known = df[df[target_dyn].notna()].copy()
print(f'Pacientes con outcome {outcome_dyn}: {len(df_known):,}')

# Para cada fase, predecir probabilidad usando el primer modelo del fold
prob_cols = {}
for fase in FASE_ORDER:
    if (outcome_dyn, fase) not in best_models:
        continue
    model_f = best_models[(outcome_dyn, fase)][0]
    feats_f = cumulative_features[fase]
    cols_f  = [c for c in feats_f if c in df_known.columns]

    X_pred = df_known[cols_f].copy()
    probs  = model_f.predict(X_pred)
    col_name = f'prob_{fase}'
    df_known[col_name] = probs
    prob_cols[fase] = col_name

print(f'Probabilidades calculadas para {len(prob_cols)} fases')
print(f'Columnas: {list(prob_cols.values())}')

Pacientes con outcome Stunting: 30,953
Probabilidades calculadas para 7 fases
Columnas: ['prob_F0_Prenatal_Parto', 'prob_F1_Nacimiento', 'prob_F2_Hospitalizacion', 'prob_F3_40semanas', 'prob_F4_3meses', 'prob_F5_6meses', 'prob_F6_9meses']


In [ ]:
# Visualizar trayectorias de riesgo individual
fases_prob = list(prob_cols.keys())
col_probs  = list(prob_cols.values())

df_traj = df_known[col_probs + [target_dyn]].dropna()

stunted    = df_traj[df_traj[target_dyn] == 1].sample(min(15, (df_traj[target_dyn]==1).sum()), random_state=42)
no_stunted = df_traj[df_traj[target_dyn] == 0].sample(min(15, (df_traj[target_dyn]==0).sum()), random_state=42)

x_fases = [fase_labels_short[f].replace('\n',' ') for f in fases_prob]
x       = np.arange(len(fases_prob))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

ax = axes[0]
for _, row in no_stunted.iterrows():
    ax.plot(x, [row[c] for c in col_probs], color='#3498db', alpha=0.3, linewidth=1)
for _, row in stunted.iterrows():
    ax.plot(x, [row[c] for c in col_probs], color='#e74c3c', alpha=0.5, linewidth=1.5)

ax.plot(x, [no_stunted[c].mean() for c in col_probs],
        color='#2980b9', linewidth=3, label='Media NO stunted', zorder=5)
ax.plot(x, [stunted[c].mean() for c in col_probs],
        color='#c0392b', linewidth=3, label='Media STUNTED', zorder=5)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.4, label='Umbral 0.5')
ax.set_xticks(x); ax.set_xticklabels(x_fases, fontsize=9)
ax.set_ylabel('P(Stunting a 12m)')
ax.set_title('Trayectorias de riesgo individual\n(rojo=stunted, azul=normal)', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)

ax2 = axes[1]
for i, (fase, col) in enumerate(zip(fases_prob, col_probs)):
    ax2.boxplot(stunted[col].values, positions=[i - 0.2], widths=0.35,
                patch_artist=True,
                boxprops=dict(facecolor='#e74c3c', alpha=0.6),
                medianprops=dict(color='darkred', linewidth=2),
                whiskerprops=dict(color='#e74c3c'), capprops=dict(color='#e74c3c'),
                flierprops=dict(marker='.', markersize=3, color='#e74c3c'))
    ax2.boxplot(no_stunted[col].values, positions=[i + 0.2], widths=0.35,
                patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.6),
                medianprops=dict(color='darkblue', linewidth=2),
                whiskerprops=dict(color='#3498db'), capprops=dict(color='#3498db'),
                flierprops=dict(marker='.', markersize=3, color='#3498db'))

ax2.axhline(0.5, color='black', linestyle='--', alpha=0.4)
ax2.set_xticks(range(len(fases_prob))); ax2.set_xticklabels(x_fases, fontsize=9)
ax2.set_title('Distribucion de riesgo por fase\n(rojo=stunted, azul=normal)', fontweight='bold')
ax2.legend(handles=[mpatches.Patch(color='#e74c3c', alpha=0.6, label='Stunted'),
                    mpatches.Patch(color='#3498db', alpha=0.6, label='Normal')], fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Sistema de Riesgo Dinamico — Stunting a 12 meses EC',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Separacion media entre grupos por fase:')
for fase, col in zip(fases_prob, col_probs):
    med_s  = stunted[col].mean()
    med_ns = no_stunted[col].mean()
    print(f'  {fase_labels_short[fase].replace(chr(10)," "):<20}: stunted={med_s:.3f}  normal={med_ns:.3f}  delta={med_s-med_ns:.3f}')

## 7. Baseline: Regresión Logística L1 (comparación)

In [19]:
# Baseline con Logistic Regression + L1 para las 156 variables universales
# Usar solo el outcome principal (Stunting) y la mejor fase

outcome_bl = 'Stunting'
fase_bl    = best_fase_per_outcome[outcome_bl][0]
X_bl, y_bl = data_store[(outcome_bl, fase_bl)]

# Imputar medianas (LR requiere datos completos)
X_bl_imp = X_bl.fillna(X_bl.median())

# Pipeline: escalado + LR L1
pipe_l1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(
                    penalty='l1', solver='liblinear',
                    C=0.1, class_weight='balanced',
                    max_iter=1000, random_state=42))
])

skf_bl = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs_bl = []
for tr, val in skf_bl.split(X_bl_imp, y_bl):
    pipe_l1.fit(X_bl_imp.iloc[tr], y_bl.iloc[tr])
    prob_val = pipe_l1.predict_proba(X_bl_imp.iloc[val])[:,1]
    aucs_bl.append(roc_auc_score(y_bl.iloc[val], prob_val))

auc_lgbm_best = best_fase_per_outcome[outcome_bl][1]
print(f'Baseline — Logistic Regression L1')
print(f'  AUC media (5-fold): {np.mean(aucs_bl):.4f} ± {np.std(aucs_bl):.4f}')
print()
print(f'LightGBM ({fase_bl})')
print(f'  AUC media (5-fold): {auc_lgbm_best:.4f}')
print()
print(f'Ganancia LightGBM vs Logistica: +{auc_lgbm_best - np.mean(aucs_bl):.4f}')

# Coeficientes no-cero de la regresión logística (variables seleccionadas por L1)
pipe_l1.fit(X_bl_imp.fillna(X_bl_imp.median()), y_bl)
coef = pd.Series(pipe_l1.named_steps['lr'].coef_[0], index=X_bl_imp.columns)
coef_nz = coef[coef != 0].sort_values(key=abs, ascending=False)
print(f'\nVariables seleccionadas por L1: {len(coef_nz)} de {len(coef)}')
print(f'Top 15 coeficientes (L1):')
for feat, val in coef_nz.head(15).items():
    signo = '+' if val > 0 else ''
    print(f'  {feat:<40} coef = {signo}{val:.4f}')

Baseline — Logistic Regression L1
  AUC media (5-fold): 0.9216 ± 0.0031

LightGBM (F6_9meses)
  AUC media (5-fold): 0.9290

Ganancia LightGBM vs Logistica: +0.0074

Variables seleccionadas por L1: 153 de 198
Top 15 coeficientes (L1):
  zscoretalla9                             coef = -1.3044
  zscoretalla6                             coef = -0.9151
  velocidad9_6mesesOMS                     coef = -0.3100
  zscorepeso6                              coef = -0.2928
  zscoretalla2                             coef = -0.2486
  zscorepeso9                              coef = -0.2273
  zscorepesotalla2                         coef = +0.1984
  vino9m                                   coef = -0.1740
  zscoretalla9cat                          coef = -0.1666
  vino6m                                   coef = -0.1625
  zscoretalla6cat                          coef = -0.1498
  zscorepesotalla9                         coef = +0.1353
  zscorepesotalla9cat                      coef = -0.1216
  ali9m     

## 8. Resumen Final del Pipeline

In [20]:
print('=' * 65)
print('RESUMEN PIPELINE DE MODELADO — Malnutricion PMCI')
print('=' * 65)

print('\nMEJOR AUC POR OUTCOME:')
for outcome, (fase, auc) in best_fase_per_outcome.items():
    _, _, spw = OUTCOMES[outcome]
    print(f'  {outcome:<12}: {auc:.4f} AUC (fase={fase})')

print('\nCASCADA TEMPORAL — AUC STUNTING:')
for fase in FASE_ORDER:
    if ('Stunting', fase) in resultados:
        mdf  = resultados[('Stunting', fase)]
        feats = len(cumulative_features[fase])
        print(f'  {fase_labels_short[fase].replace(chr(10)," "):<22}'
              f': AUC={mdf["AUC"].mean():.4f}  '
              f'Sens={mdf["Sens"].mean():.3f}  '
              f'Spec={mdf["Spec"].mean():.3f}  '
              f'({feats} features)')

print(f'''\nCONCLUSIONES:
  1. LightGBM supera a Logistica L1 en AUC
  2. La senal predictiva aumenta con cada fase (validando la cascada temporal)
  3. Ya desde F1 (nacimiento) hay capacidad predictiva util
  4. SHAP identifica los factores de riesgo mas relevantes por fase
  5. El sistema dinamico muestra separacion entre grupos desde etapas tempranas

ARCHIVOS GENERADOS:
  mod_01_auc_cascada.png       — AUC por fase y outcome
  mod_02_metricas_stunting.png — Metricas detalladas Stunting
  mod_03_roc_curvas.png        — Curvas ROC por fase
  mod_04_pr_curvas.png         — Curvas Precision-Recall
  mod_05_shap_summary.png      — SHAP top 20 factores
  mod_06_shap_importancia.png  — Importancia global SHAP
  mod_07_riesgo_dinamico.png   — Trayectorias de riesgo
''')

RESUMEN PIPELINE DE MODELADO — Malnutricion PMCI

MEJOR AUC POR OUTCOME:
  Stunting    : 0.9290 AUC (fase=F6_9meses)
  Bajo_peso   : 0.9634 AUC (fase=F6_9meses)
  Wasting     : 0.9245 AUC (fase=F6_9meses)

CASCADA TEMPORAL — AUC STUNTING:
  F0 Prenatal           : AUC=0.6454  Sens=0.509  Spec=0.689  (41 features)
  F1 Nacimiento         : AUC=0.7374  Sens=0.641  Spec=0.708  (75 features)
  F2 Hosp.              : AUC=0.7405  Sens=0.639  Spec=0.713  (107 features)
  F3 40 sem             : AUC=0.7678  Sens=0.621  Spec=0.758  (137 features)
  F4 3m                 : AUC=0.8209  Sens=0.691  Spec=0.779  (164 features)
  F5 6m                 : AUC=0.8935  Sens=0.792  Spec=0.823  (183 features)
  F6 9m                 : AUC=0.9290  Sens=0.834  Spec=0.859  (198 features)

CONCLUSIONES:
  1. LightGBM supera a Logistica L1 en AUC
  2. La senal predictiva aumenta con cada fase (validando la cascada temporal)
  3. Ya desde F1 (nacimiento) hay capacidad predictiva util
  4. SHAP identifica los fa

## 9. Análisis de Cohortes Temporales

Comparación del modelo en tres periodos históricos usando la columna `periodosanalisis`:
- **P4: 2007–2012** — cohorte más antigua
- **P5: 2013–2017** — cohorte intermedia
- **P6: 2018–2022** — cohorte más reciente

Permite detectar cambios en prevalencias, performance del modelo y evolución de protocolos de atención.
También incluye detección automática de features nuevas que aparecen en cohortes posteriores.

In [ ]:
# --- 9.1 Definición de cohortes y estadísticas descriptivas ---

COHORT_MAP = {
    '4': 'P4: 2007-2012',
    '5': 'P5: 2013-2017',
    '6': 'P6: 2018-2022',
}
COHORT_COLORS  = {'P4: 2007-2012': '#1a5276', 'P5: 2013-2017': '#27ae60', 'P6: 2018-2022': '#e74c3c'}
COHORT_MARKERS = {'P4: 2007-2012': 'o',       'P5: 2013-2017': 's',       'P6: 2018-2022': '^'}
COHORTS_ORDER  = ['P4: 2007-2012', 'P5: 2013-2017', 'P6: 2018-2022']

# periodosanalisis viene como float (4.0, 5.0...) → normalizar a entero string
df['_periodo_str'] = (df['periodosanalisis']
                      .astype(str)
                      .str.strip()
                      .str.replace(r'\.0$', '', regex=True))

# Análisis principal: periodos 4, 5, 6
df_main = df[df['_periodo_str'].isin(['4', '5', '6'])].copy()
df_main['cohort'] = df_main['_periodo_str'].map(COHORT_MAP)

print('=' * 65)
print('COHORTES TEMPORALES — Estadísticas descriptivas')
print('=' * 65)
print(f'N total (periodos 4-6): {len(df_main):,}\n')

for outcome_name, (target_col, _, _) in OUTCOMES.items():
    print(f'{outcome_name}:')
    for cohort in COHORTS_ORDER:
        sub = df_main[df_main['cohort'] == cohort][target_col].dropna()
        if len(sub) == 0:
            continue
        prev = sub.mean()
        ci   = 1.96 * np.sqrt(prev * (1 - prev) / len(sub))
        print(f'  {cohort}: n={len(sub):,}  prevalencia={prev:.1%}  IC95=[{max(0,prev-ci):.1%}, {min(1,prev+ci):.1%}]')
    print()

In [22]:
# --- 9.2 Alineación de features entre cohortes ---
# LightGBM maneja NaN nativamente: features ausentes en un cohorte se dejan como NaN

def align_features_across_cohorts(df_cohorts, features, cohort_col='cohort', min_availability=0.10):
    """
    Detecta qué features están disponibles en cada cohorte y cuáles son nuevas.
    Una feature se considera disponible si tiene > min_availability de valores no-nulos.
    Retorna (disponibles_por_cohort, nuevas_por_cohort).
    """
    cohorts = [c for c in COHORTS_ORDER if c in df_cohorts[cohort_col].unique()]
    avail, seen = {}, set()
    new_feats = {}
    for cohort in cohorts:
        sub  = df_cohorts[df_cohorts[cohort_col] == cohort]
        ok   = {f for f in features if f in sub.columns
                and sub[f].notna().mean() > min_availability}
        avail[cohort]    = ok
        new_feats[cohort] = ok - seen
        seen |= ok
    return avail, new_feats


feat_avail, feat_new = align_features_across_cohorts(
    df_main, cumulative_features['F6_9meses']
)

print('Features disponibles y nuevas por cohorte (fase F6 — todas las variables):')
print(f'{"Cohorte":<20} {"Disponibles":>12} {"Nuevas":>8}')
print('-' * 42)
for cohort in COHORTS_ORDER:
    n_avail = len(feat_avail.get(cohort, set()))
    n_new   = len(feat_new.get(cohort, set()))
    print(f'{cohort:<20} {n_avail:>12} {n_new:>8}')
    if n_new:
        sample = sorted(feat_new[cohort])[:5]
        print(f'  Primeras nuevas: {sample}{"..." if n_new > 5 else ""}')

print('\nNota: features nuevas en cohortes posteriores se incluyen en el modelo')
print('con NaN para cohortes anteriores — LightGBM las maneja sin imputación manual.')

Features disponibles y nuevas por cohorte (fase F6 — todas las variables):
Cohorte               Disponibles   Nuevas
------------------------------------------
P4: 2007-2012                   0        0
P5: 2013-2017                   0        0
P6: 2018-2022                   0        0

Nota: features nuevas en cohortes posteriores se incluyen en el modelo
con NaN para cohortes anteriores — LightGBM las maneja sin imputación manual.


In [23]:
# --- 9.3 Cascada temporal por cohorte ---
# Entrena LightGBM para cada (cohorte × fase × outcome) y guarda AUC

resultados_cohort = {}  # (cohort, outcome, fase) -> metrics_df

for cohort_name in COHORTS_ORDER:
    df_c = df_main[df_main['cohort'] == cohort_name].copy()
    print(f'\n{"="*60}')
    print(f'COHORTE: {cohort_name}  (n={len(df_c):,})')
    print('='*60)

    for outcome_name, (target_col, _, _) in OUTCOMES.items():
        for fase in FASE_ORDER:
            # Solo features disponibles en este cohorte (>5% non-null)
            feats_raw = cumulative_features[fase]
            feats_c   = [f for f in feats_raw
                         if f in df_c.columns and df_c[f].notna().mean() > 0.05]

            X, y = get_model_data(df_c, feats_c, target_col)
            if len(y) < 100 or y.sum() < 15:
                continue

            mdf, _, _ = train_lgbm_cv(X, y)
            resultados_cohort[(cohort_name, outcome_name, fase)] = mdf

            print(f'  {outcome_name:<12} {fase:<22}: '
                  f'AUC={mdf["AUC"].mean():.4f}±{mdf["AUC"].std():.4f}  '
                  f'n={len(y):,}  feats={len(feats_c)}')


COHORTE: P4: 2007-2012  (n=0)

COHORTE: P5: 2013-2017  (n=0)

COHORTE: P6: 2018-2022  (n=0)


In [ ]:
# --- 9.4 Gráficas comparativas: AUC por cohorte y prevalencias ---

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Fila 1: AUC por fase, separado por outcome
for ax, (outcome_name, (_, _, _)) in zip(axes[0], OUTCOMES.items()):
    for cohort_name in COHORTS_ORDER:
        fases_ok = [f for f in FASE_ORDER
                    if (cohort_name, outcome_name, f) in resultados_cohort]
        if not fases_ok:
            continue
        aucs = [resultados_cohort[(cohort_name, outcome_name, f)]['AUC'].mean() for f in fases_ok]
        stds = [resultados_cohort[(cohort_name, outcome_name, f)]['AUC'].std()  for f in fases_ok]
        x    = np.arange(len(fases_ok))
        ax.plot(x, aucs, marker=COHORT_MARKERS[cohort_name],
                color=COHORT_COLORS[cohort_name], linewidth=2, markersize=7, label=cohort_name)
        ax.fill_between(x, np.array(aucs)-np.array(stds), np.array(aucs)+np.array(stds),
                        alpha=0.10, color=COHORT_COLORS[cohort_name])
    x_lbls = [fase_labels_short[f] for f in fases_ok] if fases_ok else []
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    ax.set_xticks(np.arange(len(x_lbls))); ax.set_xticklabels(x_lbls, fontsize=8)
    ax.set_ylim(0.45, 1.0); ax.set_ylabel('ROC-AUC')
    ax.set_title(f'{outcome_name}\nAUC por cohorte y fase', fontweight='bold')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Fila 2: Prevalencias con IC95 por cohorte
ax_prev = axes[1][0]
x = np.arange(len(COHORTS_ORDER)); width = 0.25
for i, (outcome_name, (target_col, color, _)) in enumerate(OUTCOMES.items()):
    prevs, cis = [], []
    for cohort_name in COHORTS_ORDER:
        sub  = df_main[df_main['cohort'] == cohort_name][target_col].dropna()
        prev = sub.mean() if len(sub) > 0 else 0
        ci   = 1.96 * np.sqrt(prev * (1 - prev) / max(len(sub), 1))
        prevs.append(prev * 100); cis.append(ci * 100)
    bars = ax_prev.bar(x + i*width, prevs, width=width, color=color, alpha=0.8,
                       label=outcome_name, yerr=cis, capsize=4)
    for bar, v in zip(bars, prevs):
        ax_prev.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     f'{v:.1f}%', ha='center', va='bottom', fontsize=8)
ax_prev.set_xticks(x+width); ax_prev.set_xticklabels(COHORTS_ORDER, fontsize=8, rotation=10)
ax_prev.set_ylabel('Prevalencia (%)'); ax_prev.legend(fontsize=8)
ax_prev.set_title('Evolución de prevalencias\npor cohorte (IC 95%)', fontweight='bold')
ax_prev.grid(True, alpha=0.3, axis='y')

# Tablas resumen de AUC
for ax_t, outcome_name in zip(axes[1][1:], list(OUTCOMES.keys())[:2]):
    rows = []
    for cohort_name in COHORTS_ORDER:
        row = {'Cohorte': cohort_name}
        for fase in ['F0_Prenatal_Parto','F1_Nacimiento','F2_Hospitalizacion','F3_40semanas','F6_9meses']:
            key = (cohort_name, outcome_name, fase)
            row[fase_labels_short[fase].replace('\n',' ')] = (
                f'{resultados_cohort[key]["AUC"].mean():.3f}' if key in resultados_cohort else '—')
        rows.append(row)
    tbl_df = pd.DataFrame(rows).set_index('Cohorte')
    ax_t.axis('off')
    tbl = ax_t.table(cellText=tbl_df.values, rowLabels=tbl_df.index,
                     colLabels=tbl_df.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 1.4)
    ax_t.set_title(f'{outcome_name} — AUC por fase y cohorte', fontweight='bold')

plt.suptitle('Análisis de Cohortes Temporales — Evolución del modelo (2007–2022)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 9.5 Validación cruzada entre cohortes ---
print('=== VALIDACIÓN CRUZADA ENTRE COHORTES ===')
print('Train en un periodo → Test en otro (Stunting, F2_Hospitalizacion)\n')

outcome_cv   = 'Stunting'
target_cv    = OUTCOMES[outcome_cv][0]
fase_cv      = 'F2_Hospitalizacion'
feats_cv_raw = cumulative_features[fase_cv]
cross_auc    = {}

for train_c in COHORTS_ORDER:
    df_tr    = df_main[df_main['cohort'] == train_c]
    feats_ok = [f for f in feats_cv_raw
                if f in df_tr.columns and df_tr[f].notna().mean() > 0.05]
    X_tr, y_tr = get_model_data(df_tr, feats_ok, target_cv)
    if len(y_tr) < 100 or y_tr.sum() < 15:
        continue

    n_pos = y_tr.sum(); n_neg = len(y_tr) - n_pos
    model_cv = lgb.train(
        {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05,
         'num_leaves': 63, 'min_child_samples': 30, 'verbose': -1,
         'scale_pos_weight': round(n_neg / n_pos, 2), 'seed': 42},
        lgb.Dataset(X_tr, label=y_tr), num_boost_round=300,
    )
    for test_c in COHORTS_ORDER:
        if test_c == train_c:
            continue
        df_te = df_main[df_main['cohort'] == test_c]
        X_te, y_te = get_model_data(df_te, feats_ok, target_cv)
        if len(y_te) < 50:
            continue
        auc = roc_auc_score(y_te, model_cv.predict(X_te.reindex(columns=feats_ok, fill_value=np.nan)))
        cross_auc[(train_c, test_c)] = auc
        print(f'  Train={train_c}  →  Test={test_c}: AUC={auc:.4f}  (n_test={len(y_te):,})')

cross_mat = pd.DataFrame(index=COHORTS_ORDER, columns=COHORTS_ORDER, dtype=float)
for (tr, te), auc in cross_auc.items():
    cross_mat.loc[tr, te] = auc

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cross_mat.astype(float), annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=0.50, vmax=0.85,
            mask=cross_mat.isna(), ax=ax,
            annot_kws={'size': 13, 'weight': 'bold'}, linewidths=0.5)
ax.set_xlabel('Cohorte de prueba (test)', fontsize=11)
ax.set_ylabel('Cohorte de entrenamiento (train)', fontsize=11)
ax.set_title(f'Generalización entre cohortes — {outcome_cv}\n(fase {fase_cv})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 9.6 Detección de drift de features entre cohortes ---
fase_drift  = 'F6_9meses'
feats_drift = cumulative_features[fase_drift]

feat_avail_d, feat_new_d = align_features_across_cohorts(df_main, feats_drift)
common_feats   = set.intersection(*[feat_avail_d[c] for c in COHORTS_ORDER])
numeric_common = [f for f in common_feats if pd.api.types.is_numeric_dtype(df_main[f])]

drift_rows = []
for feat in numeric_common:
    means = {c: df_main[df_main['cohort']==c][feat].dropna().mean() for c in COHORTS_ORDER}
    vals  = [v for v in means.values() if not np.isnan(v)]
    if len(vals) >= 2:
        drift_rows.append({'feature': feat, 'drift_range': max(vals)-min(vals), **means})

drift_df = pd.DataFrame(drift_rows).sort_values('drift_range', ascending=False).reset_index(drop=True)

print(f'Features comunes a los 3 cohortes: {len(common_feats)}')
print(f'\nTop 15 features con mayor drift entre cohortes:')
cols_show = ['feature', 'drift_range'] + COHORTS_ORDER
print(drift_df[cols_show].head(15).round(4).to_string(index=False))

top10 = drift_df.head(10)
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(top10)); width = 0.28
for i, cohort_name in enumerate(COHORTS_ORDER):
    ax.bar(x + i*width, top10[cohort_name].values, width=width,
           color=COHORT_COLORS[cohort_name], alpha=0.85, label=cohort_name)
ax.set_xticks(x+width); ax.set_xticklabels(top10['feature'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Media por cohorte')
ax.set_title('Top 10 features con mayor drift entre cohortes\n'
             '(cambio de distribución = posible cambio de protocolo o población)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# --- 9.7 Análisis de sensibilidad: incluir registros 2023 con outcome completo ---

print('=== ANÁLISIS DE SENSIBILIDAD: 2007–2023 ===')
print('Solo registros con outcome de 12 meses completo\n')

# Registros 2023: periodosanalisis es nan o #NULL! pero tienen outcome válido
df_2023 = df[df['_periodo_str'].isin(['nan', '#NULL!'])].copy()
df_2023  = df_2023[df_2023['stunting12m'].notna()].copy()
df_2023['cohort'] = 'P7: 2023 (parcial)'

df_sens = pd.concat([df_main, df_2023], ignore_index=True)

print(f'Registros 2023 con outcome completo: {len(df_2023):,}')
print(f'N total análisis de sensibilidad: {len(df_sens):,}\n')

print('Distribución de cohortes:')
for cohort, grp in df_sens.groupby('cohort', sort=False):
    n_valid = grp['stunting12m'].notna().sum()
    prev = grp['stunting12m'].mean()
    print(f'  {cohort}: n={n_valid:,}  stunting={prev:.1%}')

# Comparar AUC principal vs sensibilidad (Stunting, F2)
print('\nComparación de AUC — Stunting F2_Hospitalizacion:')
feats_s = [f for f in cumulative_features['F2_Hospitalizacion']
           if f in df.columns]

for label, df_eval in [('Principal  (P4-P6, 2007-2022)', df_main),
                        ('Sensibilidad (+2023 completos)', df_sens)]:
    X_s, y_s = get_model_data(df_eval, feats_s, 'stunting12m')
    if len(y_s) > 100 and y_s.sum() > 15:
        m, _, _ = train_lgbm_cv(X_s, y_s)
        print(f'  {label}: AUC={m["AUC"].mean():.4f} ± {m["AUC"].std():.4f}  (n={len(y_s):,})')

print('\nConclusion: si los AUC son similares, los resultados son robustos.')
print('Si difieren, indica que la cohorte 2023 tiene un perfil de riesgo distinto.')

## 10. Modelado con Outcome Compuesto (4 grupos)

Entrenamiento de LightGBM multiclase para predecir el estado nutricional compuesto a 12 meses EC:
- **Grupo 0:** Normal
- **Grupo 1:** Sobrepeso/Obesidad
- **Grupo 2:** Un déficit (HAZ, WAZ o WHZ < −2)
- **Grupo 3:** Desnutrición múltiple (2 o 3 déficits simultáneos)

Se usa la misma cascada temporal F0→F6 del análisis binario.

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize

TARGET_COMP   = 'estado_nutricional_12m'
N_CLASES      = 4
CLASE_NOMBRES = [GRUPO_LABELS[i] for i in range(N_CLASES)]


def train_lgbm_multiclass_cv(X, y, n_splits=5, seed=42):
    """LightGBM multiclase con CV estratificada. Retorna métricas y predicciones OOF."""
    params = {
        'objective':        'multiclass',
        'num_class':        N_CLASES,
        'metric':           'multi_logloss',
        'learning_rate':    0.05,
        'num_leaves':       63,
        'min_child_samples': 30,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq':     5,
        'reg_alpha':        0.1,
        'reg_lambda':       0.1,
        'class_weight':     'balanced',
        'verbose':          -1,
        'seed':             seed,
    }

    skf      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics  = []
    oof_prob = np.zeros((len(y), N_CLASES))
    models   = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            params, dtrain,
            num_boost_round=500,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        )
        models.append(model)

        prob = model.predict(X_val)          # shape (n, 4)
        oof_prob[val_idx] = prob
        y_pred = prob.argmax(axis=1)

        # AUC macro one-vs-rest
        y_bin = label_binarize(y_val, classes=list(range(N_CLASES)))
        auc_macro = roc_auc_score(y_bin, prob, multi_class='ovr', average='macro')

        metrics.append({
            'fold':      fold + 1,
            'AUC_macro': round(auc_macro, 4),
            'Acc':       round((y_pred == y_val.values).mean(), 4),
            'F1_macro':  round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4),
            'F1_weighted': round(f1_score(y_val, y_pred, average='weighted', zero_division=0), 4),
            'best_iter': model.best_iteration,
        })

    return pd.DataFrame(metrics), oof_prob, models


# Cascada temporal — outcome compuesto
resultados_comp  = {}   # fase -> metrics_df
oof_probs_comp   = {}   # fase -> (prob_matrix, y_true)
best_models_comp = {}   # fase -> lista de modelos

print('CASCADA TEMPORAL — Outcome Compuesto (4 grupos)\n')
print(f'{"Fase":<24} {"AUC_macro":>10} {"Acc":>7} {"F1_macro":>9} {"F1_w":>7} {"n":>7}')
print('-' * 65)

for fase in FASE_ORDER:
    feats   = cumulative_features[fase]
    cols    = [c for c in feats if c in df.columns]
    sub     = df[cols + [TARGET_COMP]].dropna(subset=[TARGET_COMP]).copy()
    sub[TARGET_COMP] = sub[TARGET_COMP].astype(int)
    X       = sub[cols]
    y       = sub[TARGET_COMP]

    if len(y) < 200 or y.nunique() < 2:
        print(f'  {fase:<22}: insuficientes datos — omitida')
        continue

    mdf, oof_p, models = train_lgbm_multiclass_cv(X, y)

    resultados_comp [fase] = mdf
    oof_probs_comp  [fase] = (oof_p, y)
    best_models_comp[fase] = models

    print(f'  {fase:<22} {mdf["AUC_macro"].mean():>10.4f} '
          f'{mdf["Acc"].mean():>7.4f} {mdf["F1_macro"].mean():>9.4f} '
          f'{mdf["F1_weighted"].mean():>7.4f} {len(y):>7,}')

In [ ]:
# --- Métricas por clase y matriz de confusión (mejor fase) ---

# Mejor fase por AUC macro
mejor_fase_comp = max(resultados_comp, key=lambda f: resultados_comp[f]['AUC_macro'].mean())
auc_mejor       = resultados_comp[mejor_fase_comp]['AUC_macro'].mean()
print(f'Mejor fase: {mejor_fase_comp}  (AUC macro={auc_mejor:.4f})\n')

oof_p, y_true = oof_probs_comp[mejor_fase_comp]
y_pred        = oof_p.argmax(axis=1)

# Reporte por clase
print('Reporte por clase (OOF — mejor fase):')
print(classification_report(y_true, y_pred,
                             target_names=CLASE_NOMBRES,
                             zero_division=0))

# AUC por clase (one-vs-rest)
y_bin = label_binarize(y_true, classes=list(range(N_CLASES)))
print('AUC por clase (one-vs-rest):')
for i, nombre in enumerate(CLASE_NOMBRES):
    if y_bin[:, i].sum() > 0:
        auc_i = roc_auc_score(y_bin[:, i], oof_p[:, i])
        print(f'  {nombre:<28}: AUC = {auc_i:.4f}')

# Gráficas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Evolución AUC macro por fase
ax1 = axes[0]
fases_ok = [f for f in FASE_ORDER if f in resultados_comp]
aucs_m   = [resultados_comp[f]['AUC_macro'].mean() for f in fases_ok]
stds_m   = [resultados_comp[f]['AUC_macro'].std()  for f in fases_ok]
x        = np.arange(len(fases_ok))
ax1.plot(x, aucs_m, marker='o', linewidth=2.5, markersize=8, color='#8e44ad')
ax1.fill_between(x, np.array(aucs_m)-np.array(stds_m),
                    np.array(aucs_m)+np.array(stds_m), alpha=0.15, color='#8e44ad')
for xi, v in zip(x, aucs_m):
    ax1.text(xi, v+0.008, f'{v:.3f}', ha='center', fontsize=9,
             fontweight='bold', color='#8e44ad')
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='Aleatorio')
ax1.set_xticks(x)
ax1.set_xticklabels([fase_labels_short[f] for f in fases_ok], fontsize=8)
ax1.set_ylim(0.45, 1.0); ax1.set_ylabel('AUC macro (OvR)')
ax1.set_title('AUC macro por fase\nOutcome compuesto (4 grupos)', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# Panel 2: Matriz de confusión normalizada
ax2 = axes[1]
cm   = confusion_matrix(y_true, y_pred, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASE_NOMBRES)
disp.plot(ax=ax2, colorbar=False, cmap='Blues', values_format='.2f')
ax2.set_title(f'Matriz de confusión normalizada\n{mejor_fase_comp}', fontweight='bold')
ax2.set_xticklabels(CLASE_NOMBRES, rotation=20, ha='right', fontsize=8)
ax2.set_yticklabels(CLASE_NOMBRES, fontsize=8)

# Panel 3: F1 por clase y fase
ax3 = axes[2]
from sklearn.metrics import f1_score as f1s
f1_por_clase = {}
for clase_id, clase_nombre in enumerate(CLASE_NOMBRES):
    f1_por_clase[clase_nombre] = []
    for f in fases_ok:
        op, yt = oof_probs_comp[f]
        yp_f   = op.argmax(axis=1)
        f1_c   = f1s((yt == clase_id).astype(int),
                     (yp_f == clase_id).astype(int), zero_division=0)
        f1_por_clase[clase_nombre].append(f1_c)

clase_colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']
for (nombre, vals), col in zip(f1_por_clase.items(), clase_colors):
    ax3.plot(x, vals, marker='o', linewidth=2, markersize=6, color=col, label=nombre)
ax3.set_xticks(x)
ax3.set_xticklabels([fase_labels_short[f] for f in fases_ok], fontsize=8)
ax3.set_ylim(0, 1.05); ax3.set_ylabel('F1-score')
ax3.set_title('F1 por clase y fase', fontweight='bold')
ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3)

plt.suptitle('Modelado Multiclase — Estado Nutricional Compuesto a 12 meses EC',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP multiclase — factores de riesgo por grupo nutricional ---

fase_shap_comp  = mejor_fase_comp
feats_shap      = cumulative_features[fase_shap_comp]
cols_shap       = [c for c in feats_shap if c in df.columns]
sub_shap        = df[cols_shap + [TARGET_COMP]].dropna(subset=[TARGET_COMP]).copy()
sub_shap[TARGET_COMP] = sub_shap[TARGET_COMP].astype(int)
X_shap_comp     = sub_shap[cols_shap]

model_shap_comp = best_models_comp[fase_shap_comp][0]
X_sample_comp   = X_shap_comp.sample(min(2000, len(X_shap_comp)), random_state=42)

print(f'Calculando SHAP multiclase para {fase_shap_comp}...')
explainer_comp = shap.TreeExplainer(model_shap_comp)
shap_raw       = explainer_comp.shap_values(X_sample_comp)

# Normalizar formato: SHAP puede devolver lista o array 3D según versión
if isinstance(shap_raw, list):
    sv_por_clase = shap_raw                              # lista de (n_samples, n_features)
else:
    sv_por_clase = [shap_raw[:, :, i] for i in range(N_CLASES)]  # 3D → lista

print(f'  Clases: {len(sv_por_clase)}  |  Shape por clase: {np.array(sv_por_clase[0]).shape}')

# Top 15 features por clase
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for clase_id, (nombre, ax) in enumerate(zip(CLASE_NOMBRES, axes.flatten())):
    sv  = np.abs(sv_por_clase[clase_id])
    imp = pd.DataFrame({
        'feature':       X_sample_comp.columns,
        'mean_abs_shap': sv.mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False).head(15)

    color = clase_colors[clase_id]
    bars  = ax.barh(imp['feature'][::-1], imp['mean_abs_shap'][::-1],
                    color=color, alpha=0.8, edgecolor='white')
    for bar, v in zip(bars, imp['mean_abs_shap'][::-1]):
        ax.text(v + 0.0001, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=7)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'Grupo {clase_id}: {nombre}\nTop 15 factores', fontweight='bold', color=color)
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle(f'SHAP por grupo nutricional — {fase_shap_comp}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla comparativa top 5 por grupo
print('\nTop 5 features por grupo nutricional (★):')
print(f'{"Feature":<35}', end='')
for n in CLASE_NOMBRES:
    print(f'{n[:12]:>14}', end='')
print()
print('-' * (35 + 14 * N_CLASES))

top5_by_class, all_top = {}, set()
for clase_id, nombre in enumerate(CLASE_NOMBRES):
    imp = pd.Series(np.abs(sv_por_clase[clase_id]).mean(axis=0),
                    index=X_sample_comp.columns).nlargest(5)
    top5_by_class[nombre] = imp
    all_top |= set(imp.index)

for feat in sorted(all_top):
    print(f'{feat:<35}', end='')
    for nombre in CLASE_NOMBRES:
        v    = top5_by_class[nombre].get(feat, 0)
        mark = '★' if feat in top5_by_class[nombre].index else ''
        print(f'{mark+str(round(v,4)):>14}', end='')
    print()


## 11. Split Train/Test y Exportación para Dashboard

División aleatoria 80/20 estratificada por outcome principal (Stunting).
El set de prueba se usa para evaluar el modelo final y generar los archivos
que consume el dashboard interactivo.

In [ ]:
from sklearn.model_selection import train_test_split

DASHBOARD_DIR = 'dashboard_data'
os.makedirs(DASHBOARD_DIR, exist_ok=True)

# Split sobre registros con outcome principal disponible
df_valido = df[df['stunting12m'].notna()].copy()

idx_train, idx_test = train_test_split(
    df_valido.index,
    test_size=0.20,
    random_state=42,
    stratify=df_valido['stunting12m'].astype(int)
)

df_train = df.loc[idx_train].copy()
df_test  = df.loc[idx_test].copy()

print('=' * 55)
print('SPLIT TRAIN / TEST  (80% / 20%  —  aleatorio estratificado)')
print('=' * 55)
print(f'  Train : {len(df_train):,} registros')
print(f'  Test  : {len(df_test):,} registros')
print()

# Verificar balance de outcomes en cada split
for outcome_name, (target_col, _, _) in OUTCOMES.items():
    tr_prev = df_train[target_col].mean()
    te_prev = df_test [target_col].mean()
    print(f'  {outcome_name:<12}  train={tr_prev:.1%}  test={te_prev:.1%}')

In [ ]:
# --- Entrenar modelos finales sobre train y evaluar en test ---

metricas_test  = []   # para metricas_por_fase.json
modelos_finales = {}  # (outcome, fase) -> modelo entrenado en train

print('Entrenando sobre TRAIN y evaluando en TEST...\n')
print(f'{"Outcome":<12} {"Fase":<24} {"AUC_test":>9} {"Sens":>7} {"Spec":>7} {"n_test":>7}')
print('-' * 65)

for outcome_name, (target_col, _, _) in OUTCOMES.items():
    for fase in FASE_ORDER:
        feats = cumulative_features[fase]

        # Train
        cols_tr  = [c for c in feats if c in df_train.columns]
        X_tr, y_tr = get_model_data(df_train, cols_tr, target_col)
        if len(y_tr) < 100 or y_tr.sum() < 15:
            continue

        n_pos = y_tr.sum(); n_neg = len(y_tr) - n_pos
        params_f = {**{
            'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05,
            'num_leaves': 63, 'min_child_samples': 30, 'feature_fraction': 0.8,
            'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.1,
            'reg_lambda': 0.1, 'verbose': -1, 'seed': 42,
            'scale_pos_weight': round(n_neg / n_pos, 2)
        }}

        # Usar best_iteration promedio de los folds CV como referencia
        key_cv = (outcome_name, fase)
        n_rounds = 300
        if key_cv in best_models:
            n_rounds = max(50, int(np.mean([m.best_iteration for m in best_models[key_cv]])))

        model_f = lgb.train(params_f, lgb.Dataset(X_tr, label=y_tr),
                            num_boost_round=n_rounds)
        modelos_finales[(outcome_name, fase)] = model_f

        # Test
        cols_te = [c for c in feats if c in df_test.columns]
        X_te, y_te = get_model_data(df_test, cols_te, target_col)
        if len(y_te) < 20:
            continue

        X_te_al = X_te.reindex(columns=cols_tr, fill_value=np.nan)
        prob_te  = model_f.predict(X_te_al)
        auc_te   = roc_auc_score(y_te, prob_te)
        pred_bin = (prob_te >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, pred_bin).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0

        metricas_test.append({
            'outcome': outcome_name, 'fase': fase,
            'AUC_test': round(auc_te, 4),
            'Sens_test': round(sens, 4), 'Spec_test': round(spec, 4),
            'n_test': int(len(y_te)), 'n_pos_test': int(y_te.sum()),
        })
        print(f'  {outcome_name:<12} {fase:<24} {auc_te:>9.4f} {sens:>7.3f} {spec:>7.3f} {len(y_te):>7,}')

# Guardar métricas
with open(f'{DASHBOARD_DIR}/metricas_por_fase.json', 'w') as f:
    json.dump(metricas_test, f, indent=2, ensure_ascii=False)
print(f'\nGuardado: {DASHBOARD_DIR}/metricas_por_fase.json')

In [ ]:
# --- Generar predicciones completas sobre el test set ---

print('Generando predicciones sobre test set...')
df_pred = pd.DataFrame(index=df_test.index)

# ID de paciente si existe
if 'Idenfinal' in df_test.columns:
    df_pred['Idenfinal'] = df_test['Idenfinal']

# Outcomes reales
for outcome_name, (target_col, _, _) in OUTCOMES.items():
    df_pred[f'real_{outcome_name}'] = df_test[target_col]
df_pred['real_estado_nutricional'] = df_test[TARGET_COMP]

# Probabilidades por fase y outcome
for outcome_name, (target_col, _, _) in OUTCOMES.items():
    for fase in FASE_ORDER:
        key = (outcome_name, fase)
        if key not in modelos_finales:
            continue
        model_f = modelos_finales[key]
        feats   = cumulative_features[fase]
        cols_tr = [c for c in feats if c in df_train.columns]
        X_te_al = df_test.reindex(columns=cols_tr, fill_value=np.nan)
        df_pred[f'prob_{outcome_name}_{fase}'] = model_f.predict(X_te_al)

# Variables clínicas de contexto
vars_contexto = [
    'Iden_FechaParto', '_periodo_str', 'cohort',
    'ERN_Peso', 'ERN_Talla', 'ERN_EdadGestacional',
    'CP_TallaMadre', 'Iden_Sede',
]
for v in vars_contexto:
    if v in df_test.columns:
        df_pred[v] = df_test[v]

df_pred.to_csv(f'{DASHBOARD_DIR}/test_predictions.csv', index=True)
print(f'Guardado: {DASHBOARD_DIR}/test_predictions.csv')
print(f'  Shape : {df_pred.shape}  ({len(df_pred):,} pacientes × {df_pred.shape[1]} columnas)')
print(f'  Cols probabilidad : {sum(1 for c in df_pred.columns if c.startswith("prob_"))}')
print(f'  Cols outcome real : {sum(1 for c in df_pred.columns if c.startswith("real_"))}')

In [ ]:
# --- SHAP sobre test set ---

print('Calculando SHAP sobre muestra del test set...')

fase_shap_dash    = 'F2_Hospitalizacion'
outcome_shap_dash = 'Stunting'
key_shap_dash     = (outcome_shap_dash, fase_shap_dash)

if key_shap_dash in modelos_finales:
    model_shap_d = modelos_finales[key_shap_dash]
    cols_d       = [c for c in cumulative_features[fase_shap_dash] if c in df_train.columns]
    X_shap_samp  = df_test.reindex(columns=cols_d, fill_value=np.nan).sample(
                       min(1500, len(df_test)), random_state=42)

    shap_vals_d  = shap.TreeExplainer(model_shap_d).shap_values(X_shap_samp)

    # SHAP por paciente
    df_shap = pd.DataFrame(shap_vals_d, columns=cols_d, index=X_shap_samp.index)
    df_shap.insert(0, 'paciente_idx', X_shap_samp.index)
    df_shap.to_csv(f'{DASHBOARD_DIR}/shap_values.csv', index=False)
    print(f'Guardado: {DASHBOARD_DIR}/shap_values.csv  ({df_shap.shape})')

    # Importancia global
    pd.DataFrame({
        'feature':       cols_d,
        'mean_abs_shap': np.abs(shap_vals_d).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False).to_csv(
        f'{DASHBOARD_DIR}/shap_importancia_global.csv', index=False)
    print(f'Guardado: {DASHBOARD_DIR}/shap_importancia_global.csv')

# --- Cohort stats ---
cohort_stats = []
if 'cohort' in df.columns:
    for cohort_name, grp in df[df['cohort'].notna()].groupby('cohort'):
        row = {'cohort': cohort_name, 'n_total': len(grp)}
        for outcome_name, (target_col, _, _) in OUTCOMES.items():
            sub = grp[target_col].dropna()
            row[f'prev_{outcome_name}'] = round(sub.mean(), 4) if len(sub) > 0 else None
            row[f'n_{outcome_name}']    = int(sub.notna().sum())
        cohort_stats.append(row)

pd.DataFrame(cohort_stats).to_csv(f'{DASHBOARD_DIR}/cohort_stats.csv', index=False)
with open(f'{DASHBOARD_DIR}/cohort_stats.json', 'w', encoding='utf-8') as f:
    json.dump(cohort_stats, f, indent=2, ensure_ascii=False)
print(f'Guardado: {DASHBOARD_DIR}/cohort_stats.csv  (y .json)')

# --- Métricas también en CSV ---
pd.DataFrame(metricas_test).to_csv(f'{DASHBOARD_DIR}/metricas_por_fase.csv', index=False)
print(f'Guardado: {DASHBOARD_DIR}/metricas_por_fase.csv')

# --- Resumen ---
print('\n' + '=' * 55)
print('ARCHIVOS LISTOS PARA EL DASHBOARD')
print('=' * 55)
archivos = [
    'test_predictions.csv',
    'metricas_por_fase.csv',
    'shap_importancia_global.csv',
    'shap_values.csv',
    'cohort_stats.csv',
    'cohort_stats.json',
    'metricas_por_fase.json',
]
for fname in archivos:
    fpath = f'{DASHBOARD_DIR}/{fname}'
    size  = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    print(f'  {fname:<42} {size:>7.1f} KB')
print(f'\nDirectorio: {os.path.abspath(DASHBOARD_DIR)}/')

In [ ]:
# --- README.json para el equipo del dashboard ---

readme = {
    "proyecto": "Predicción de Malnutrición a 12 meses EC — PMCI Fundación Canguro",
    "generado": datetime.now().strftime('%Y-%m-%d %H:%M'),
    "split": "80% train / 20% test  (aleatorio estratificado, random_state=42)",
    "como_leer_csv": "pd.read_csv('dashboard_data/nombre_archivo.csv')",
    "fases": {
        "F0_Prenatal_Parto":  "Variables prenatales y del parto (41 features)",
        "F1_Nacimiento":      "+ datos del nacimiento (75 features)",
        "F2_Hospitalizacion": "+ datos de hospitalización (107 features)",
        "F3_40semanas":       "+ visita 40 semanas EC (137 features)",
        "F4_3meses":          "+ visita 3 meses EC (164 features)",
        "F5_6meses":          "+ visita 6 meses EC (183 features)",
        "F6_9meses":          "+ visita 9 meses EC (198 features)",
    },
    "outcomes": {
        "Stunting":  "Retraso en crecimiento — HAZ < -2 DS",
        "Bajo_peso": "Bajo peso para la edad — WAZ < -2 DS",
        "Wasting":   "Desnutrición aguda — WHZ < -2 DS",
    },
    "grupos_nutricionales": {
        "0": "Normal — ningún indicador < -2 DS",
        "1": "Sobrepeso/Obesidad — sin desnutrición",
        "2": "Un déficit — exactamente 1 de HAZ/WAZ/WHZ < -2 DS",
        "3": "Desnutrición múltiple — 2 o 3 indicadores < -2 DS",
    },
    "archivos": {
        "test_predictions.csv": {
            "descripcion": "Predicciones del modelo sobre el 20% de test. Una fila por paciente.",
            "columnas_clave": {
                "real_Stunting":           "Outcome real Stunting (0/1, puede ser NaN)",
                "real_Bajo_peso":          "Outcome real Bajo peso (0/1, puede ser NaN)",
                "real_Wasting":            "Outcome real Wasting (0/1, puede ser NaN)",
                "real_estado_nutricional": "Grupo nutricional real (0-3, puede ser NaN)",
                "prob_Stunting_F0_*":      "Probabilidad de Stunting en cada fase (0.0 a 1.0)",
                "prob_Bajo_peso_F0_*":     "Probabilidad de Bajo peso en cada fase",
                "prob_Wasting_F0_*":       "Probabilidad de Wasting en cada fase",
                "ERN_Peso":                "Peso al nacer (gramos)",
                "ERN_Talla":               "Talla al nacer (cm)",
                "ERN_EdadGestacional":     "Edad gestacional al nacer (semanas)",
                "CP_TallaMadre":           "Talla de la madre (cm)",
                "Iden_Sede":               "Sede de atención",
                "cohort":                  "Cohorte temporal (P4/P5/P6)",
            },
            "nota": "Para cascada de riesgo individual: usar columnas prob_*_F0 hasta prob_*_F6"
        },
        "metricas_por_fase.csv": {
            "descripcion": "AUC, Sensibilidad y Especificidad en test por fase y outcome.",
            "columnas": ["outcome", "fase", "AUC_test", "Sens_test", "Spec_test", "n_test", "n_pos_test"]
        },
        "shap_importancia_global.csv": {
            "descripcion": "Ranking de importancia de features (Stunting, F2_Hospitalizacion).",
            "columnas": {
                "feature":       "Nombre de la variable clínica",
                "mean_abs_shap": "Importancia media — mayor = más influyente",
            },
            "nota": "Ordenado de mayor a menor. Usar para ranking de factores de riesgo."
        },
        "shap_values.csv": {
            "descripcion": "Valores SHAP individuales por paciente (muestra 1500 del test set).",
            "columnas": "paciente_idx + una columna por feature",
            "interpretacion": "SHAP > 0 aumenta riesgo, SHAP < 0 lo disminuye",
            "nota": "Filtrar por paciente_idx y ordenar por |valor| para explicación individual."
        },
        "cohort_stats.csv": {
            "descripcion": "Prevalencias de malnutrición por cohorte temporal.",
            "columnas": ["cohort", "n_total", "prev_Stunting", "prev_Bajo_peso", "prev_Wasting"]
        },
    },
    "modelos": {
        "directorio": "modelos_pmci/",
        "formato": "LightGBM nativo (.lgb)",
        "como_cargar": "lgb.Booster(model_file='modelos_pmci/modelo_Stunting_F2_Hospitalizacion.lgb')",
        "features": "Cargar modelos_pmci/features_{fase}.json antes de predecir",
        "total": 21,
        "convencion": "modelo_{outcome}_{fase}.lgb"
    }
}

with open(f'{DASHBOARD_DIR}/README.json', 'w', encoding='utf-8') as f:
    json.dump(readme, f, indent=2, ensure_ascii=False)
print(f'Guardado: {DASHBOARD_DIR}/README.json')
print('\nEstructura final dashboard_data/:')
for fname in sorted(os.listdir(DASHBOARD_DIR)):
    size = os.path.getsize(f'{DASHBOARD_DIR}/{fname}') / 1024
    print(f'  {fname:<45} {size:>7.1f} KB')

## 10. Guardar Modelos Entrenados

Persiste a disco los modelos finales (reentrenados en el 100% de los datos) y los feature sets por fase.
Estructura guardada en `modelos_pmci/`:
- `modelo_{outcome}_{fase}.lgb` — modelo LightGBM nativo
- `features_{fase}.json` — lista de features de esa fase
- `metadata.json` — AUC de referencia, fecha, parámetros

In [ ]:
import os
from datetime import datetime

MODELS_DIR = 'modelos_pmci'
os.makedirs(MODELS_DIR, exist_ok=True)

PARAMS_FINAL = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         -1,
    'min_child_samples': 30,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'verbose':           -1,
    'seed':              42,
}

metadata = {'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'), 'modelos': []}

print('Entrenando modelos finales (100% datos) y guardando...\n')

for outcome_name, (target_col, _, _) in OUTCOMES.items():
    for fase in FASE_ORDER:
        feats = cumulative_features[fase]
        X, y  = get_model_data(df, feats, target_col)

        if len(y) < 200 or y.sum() < 20:
            continue

        n_pos = y.sum(); n_neg = len(y) - n_pos
        params = {**PARAMS_FINAL, 'scale_pos_weight': round(n_neg / n_pos, 2)}

        # Número de iteraciones = media de best_iteration de los folds CV
        best_iters = [m.best_iteration for m in best_models[(outcome_name, fase)]]
        n_rounds   = max(50, int(np.mean(best_iters)))

        model_final = lgb.train(params, lgb.Dataset(X, label=y), num_boost_round=n_rounds)

        # Guardar modelo en formato nativo LightGBM
        safe_fase   = fase.replace('/', '_')
        model_path  = f'{MODELS_DIR}/modelo_{outcome_name}_{safe_fase}.lgb'
        model_final.save_model(model_path)

        # Guardar feature list de esta fase
        feat_path = f'{MODELS_DIR}/features_{safe_fase}.json'
        with open(feat_path, 'w') as f_out:
            json.dump({'fase': fase, 'features': list(X.columns)}, f_out, indent=2)

        auc_ref = resultados[(outcome_name, fase)]['AUC'].mean()
        metadata['modelos'].append({
            'outcome':   outcome_name,
            'fase':      fase,
            'n_samples': int(len(y)),
            'n_pos':     int(n_pos),
            'auc_cv':    round(auc_ref, 4),
            'n_rounds':  n_rounds,
            'model_file': model_path,
            'feat_file':  feat_path,
        })
        print(f'  {outcome_name:<12} {fase:<22}: guardado  (n={len(y):,}, AUC_cv={auc_ref:.4f}, iter={n_rounds})')

# Guardar metadata global
meta_path = f'{MODELS_DIR}/metadata.json'
with open(meta_path, 'w') as f_out:
    json.dump(metadata, f_out, indent=2, ensure_ascii=False)

print(f'\nTotal modelos guardados: {len(metadata["modelos"])}')
print(f'Directorio: {os.path.abspath(MODELS_DIR)}/')
print(f'Metadata  : {meta_path}')

## 11. Inferencia — Predicción sobre paciente nuevo

Función reutilizable `predecir_riesgo()` que carga el modelo guardado y predice para un paciente dado.
Funciona con un diccionario de variables o un DataFrame de una fila.
Features ausentes o nuevas se manejan automáticamente con NaN.

In [ ]:
def predecir_riesgo(paciente, fase, outcome='Stunting', models_dir='modelos_pmci',
                    umbral=0.5, verbose=True):
    """
    Predice riesgo de malnutrición a 12 meses EC para un paciente nuevo.

    Parámetros
    ----------
    paciente   : dict o pd.Series o pd.DataFrame (1 fila) con variables clínicas
    fase       : str — una de F0_Prenatal_Parto ... F6_9meses
    outcome    : str — 'Stunting', 'Bajo_peso' o 'Wasting'
    models_dir : str — directorio con los modelos guardados
    umbral     : float — umbral de clasificación (default 0.5)
    verbose    : bool — imprime resumen

    Retorna
    -------
    dict con probabilidad, clasificación, features usadas/faltantes
    """
    safe_fase  = fase.replace('/', '_')
    model_path = f'{models_dir}/modelo_{outcome}_{safe_fase}.lgb'
    feat_path  = f'{models_dir}/features_{safe_fase}.json'

    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Modelo no encontrado: {model_path}\n'
                                f'Ejecuta primero la sección 10 para guardar los modelos.')

    model    = lgb.Booster(model_file=model_path)
    with open(feat_path) as f:
        features = json.load(f)['features']

    # Convertir a DataFrame de 1 fila
    if isinstance(paciente, dict):
        df_pac = pd.DataFrame([paciente])
    elif isinstance(paciente, pd.Series):
        df_pac = paciente.to_frame().T
    else:
        df_pac = paciente.copy()

    # Alinear features: columnas faltantes → NaN, columnas extra → ignoradas
    features_presentes = [f for f in features if f in df_pac.columns]
    features_ausentes  = [f for f in features if f not in df_pac.columns]
    features_nuevas    = [c for c in df_pac.columns if c not in features]

    X = df_pac.reindex(columns=features, fill_value=np.nan)

    probabilidad  = float(model.predict(X)[0])
    clasificacion = 'RIESGO' if probabilidad >= umbral else 'SIN RIESGO'

    if verbose:
        print(f'{"="*55}')
        print(f'PREDICCIÓN DE RIESGO — {outcome} a 12 meses EC')
        print(f'{"="*55}')
        print(f'  Fase evaluada   : {fase}')
        print(f'  Probabilidad    : {probabilidad:.1%}')
        print(f'  Clasificación   : {clasificacion}  (umbral={umbral:.0%})')
        print(f'  Features usadas : {len(features_presentes)} / {len(features)}')
        if features_ausentes:
            print(f'  Features faltantes ({len(features_ausentes)}): '
                  f'{features_ausentes[:5]}{"..." if len(features_ausentes)>5 else ""}')
        if features_nuevas:
            print(f'  Features nuevas ignoradas ({len(features_nuevas)}): '
                  f'{features_nuevas[:5]}{"..." if len(features_nuevas)>5 else ""}')

    return {
        'probabilidad':       probabilidad,
        'clasificacion':      clasificacion,
        'outcome':            outcome,
        'fase':               fase,
        'features_usadas':    len(features_presentes),
        'features_faltantes': features_ausentes,
        'features_nuevas':    features_nuevas,
    }


print('Función predecir_riesgo() definida.')

In [ ]:
# --- Demo 1: paciente real del dataset (tomado por índice) ---
paciente_real = df.iloc[42].to_dict()

print('=== DEMO 1: Paciente real del dataset ===')
for fase_demo in ['F1_Nacimiento', 'F2_Hospitalizacion', 'F4_3meses', 'F6_9meses']:
    res = predecir_riesgo(paciente_real, fase=fase_demo, outcome='Stunting', verbose=False)
    print(f'  {fase_demo:<22}: P={res["probabilidad"]:.1%}  → {res["clasificacion"]}')

# Verificar outcome real
outcome_real = paciente_real.get('stunting12m')
print(f'\n  Outcome real (stunting12m): {"STUNTED" if outcome_real == 1 else "NO stunted" if outcome_real == 0 else "desconocido"}')

# --- Demo 2: predicción en cascada completa con gráfica ---
print('\n=== DEMO 2: Cascada de riesgo para el mismo paciente ===')
fases_cascade = [f for f in FASE_ORDER
                 if os.path.exists(f'modelos_pmci/modelo_Stunting_{f}.lgb')]
probs_cascade  = []

for fase_c in fases_cascade:
    res = predecir_riesgo(paciente_real, fase=fase_c, outcome='Stunting', verbose=False)
    probs_cascade.append(res['probabilidad'])
    print(f'  {fase_c:<22}: {res["probabilidad"]:.1%}')

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(fases_cascade))
color_line = '#e74c3c' if (outcome_real == 1) else '#3498db'
ax.plot(x, probs_cascade, marker='o', linewidth=2.5, markersize=9, color=color_line)
for xi, p in zip(x, probs_cascade):
    ax.text(xi, p + 0.025, f'{p:.1%}', ha='center', fontsize=10, fontweight='bold', color=color_line)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Umbral 0.5')
ax.fill_between(x, 0.5, probs_cascade,
                where=[p >= 0.5 for p in probs_cascade],
                alpha=0.15, color='#e74c3c', label='Zona de riesgo')
ax.set_xticks(x)
ax.set_xticklabels([fase_labels_short[f] for f in fases_cascade], fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('P(Stunting a 12m EC)')
ax.set_title(f'Cascada de riesgo — Paciente demo\n'
             f'Outcome real: {"STUNTED" if outcome_real==1 else "NO stunted" if outcome_real==0 else "desconocido"}',
             fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Demo 3: paciente con solo variables de nacimiento (F1) ---
print('\n=== DEMO 3: Paciente nuevo con solo datos de nacimiento ===')
paciente_nuevo = {
    'CP_TallaMadre':    158.0,
    'ERN_Peso':         1050.0,   # peso al nacer (gramos)
    'ERN_Talla':        35.0,     # talla al nacer (cm)
    'ERN_EdadGestacional': 28,    # semanas de gestación
    'CP_Embarazos':     2,
}
res_nuevo = predecir_riesgo(paciente_nuevo, fase='F1_Nacimiento', outcome='Stunting')